# 📈 US Stock Price Collector

Fetches full daily OHLCV history (2010 → today) for US stocks using **yfinance** (Yahoo Finance, free, no API key).
Filters to tickers that have **ever traded above $20**, then stores results on GitHub.

## Universe
Starts with ~1,200 tickers from S&P 500 + NASDAQ 100 + Russell 1000, then keeps only those with max close ≥ $20.

## Architecture
```
Yahoo Finance (yfinance)  →  this notebook  →  GitHub (CSV per ticker + master index)
```

## Files written to GitHub
| File | Contents |
|------|----------|
| `stocks/prices_{TICKER}.csv` | Daily OHLCV for one ticker |
| `stocks/index.csv` | Master index: ticker, name, sector, max_price, row_count, last_updated |

---
## 0. Install dependencies

---
## 1. Configuration

In [1]:
import os
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()

# ── GitHub settings (same repo as HN data) ────────────────────────────────
GITHUB_REPO   = "annhmartin/dataviz-historical-stocks-AnnetteMartin"
GITHUB_TOKEN  = os.environ.get("GITHUB_TOKEN", None)
STOCKS_PREFIX = "stocks"           # folder inside the repo

# ── Filter settings ───────────────────────────────────────────────────────
MIN_PRICE     = 20.0               # keep tickers where max(close) >= this
START_DATE    = "2010-01-01"       # match HN historical window
END_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")

print("Configuration loaded ✓")
print(f"  Repo        : {GITHUB_REPO}")
print(f"  Token       : {'set ✓' if GITHUB_TOKEN else 'NOT SET'}")
print(f"  Price filter: max close >= ${MIN_PRICE}")
print(f"  Date range  : {START_DATE} → {END_DATE}")

Configuration loaded ✓
  Repo        : annhmartin/dataviz-historical-stocks-AnnetteMartin
  Token       : set ✓
  Price filter: max close >= $20.0
  Date range  : 2010-01-01 → 2026-06-28


---
## 2. Build the ticker universe

Pulls S&P 500 from Wikipedia, NASDAQ 100 and Russell 1000 from known sources, deduplicates.

In [2]:
import requests
import pandas as pd
import io


def _fetch_html(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36"
    }
    resp = requests.get(url, headers=headers, timeout=20)
    resp.raise_for_status()
    return resp.text


def get_all_us_tickers() -> pd.DataFrame:
    frames = []

    for exchange in ["nasdaq", "nyse", "amex"]:
        url  = f"https://www.nasdaqtrader.com/dynamic/SymDir/{exchange}listed.txt"
        resp = requests.get(url, timeout=20)
        resp.raise_for_status()

        rows = []
        lines = resp.text.strip().split("\n")
        for line in lines[1:]:  # skip header
            parts = line.split("|")
            if len(parts) >= 2:
                ticker = parts[0].strip()
                name   = parts[1].strip()
                rows.append({"ticker": ticker, "name": name,
                             "sector": exchange.upper(), "source": exchange.upper()})

        df = pd.DataFrame(rows)

        # Filter junk symbols
        df = df[~df["ticker"].str.contains(r'[\$\^\+\*\#]', regex=True, na=False)]
        df = df[df["ticker"].str.len() <= 5]
        df = df[df["ticker"].str.match(r'^[A-Z]+$')]
        df = df[df["ticker"].notna()]
        df = df[df["ticker"] != "nan"]

        frames.append(df)
        print(f"  {exchange.upper()}: {len(df)} tickers")

    universe = (
        pd.concat(frames, ignore_index=True)
        .drop_duplicates(subset="ticker")
        .reset_index(drop=True)
    )
    return universe


print("Fetching all US tickers …")
universe = get_all_us_tickers()
print(f"\n✓ Total universe: {len(universe)} tickers")
universe.head()


print("Fetching all US tickers …")
universe = get_all_us_tickers()
print(f"\n✓ Total universe: {len(universe)} tickers")
universe.head()

Fetching all US tickers …
  NASDAQ: 5533 tickers
  NYSE: 0 tickers
  AMEX: 0 tickers

✓ Total universe: 5533 tickers
Fetching all US tickers …
  NASDAQ: 5533 tickers
  NYSE: 0 tickers
  AMEX: 0 tickers

✓ Total universe: 5533 tickers


,ticker,name,sector,source
0,AAAP,Pacer Barings CLO Market Flex ETF,NASDAQ,NASDAQ
1,AACB,Artius II Acquisition Inc. - Class A Ordinary ...,NASDAQ,NASDAQ
2,AACBR,Artius II Acquisition Inc. - Rights,NASDAQ,NASDAQ
3,AACBU,Artius II Acquisition Inc. - Units,NASDAQ,NASDAQ
4,AACG,ATA Creativity Global - American Depositary Sh...,NASDAQ,NASDAQ


---
## 3. GitHub helpers

In [3]:
import base64
import io
import time

GITHUB_API = "https://api.github.com"


def _gh_headers(token: str) -> dict:
    return {
        "Authorization"       : f"Bearer {token}",
        "Accept"              : "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }


def push_csv_to_github(
    df: pd.DataFrame,
    repo: str,
    path: str,
    token: str,
    commit_message: str,
) -> None:
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    encoded = base64.b64encode(buf.getvalue().encode()).decode()

    url     = f"{GITHUB_API}/repos/{repo}/contents/{path}"
    headers = _gh_headers(token)

    existing_sha = None
    check = requests.get(url, headers=headers, timeout=15)
    if check.status_code == 200:
        existing_sha = check.json().get("sha")

    payload = {"message": commit_message, "content": encoded}
    if existing_sha:
        payload["sha"] = existing_sha

    resp = requests.put(url, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()


def load_csv_from_github(repo: str, path: str, token: str | None = None) -> pd.DataFrame:
    """Load via raw URL — handles large files."""
    raw_url = f"https://raw.githubusercontent.com/{repo}/main/{path}"
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    resp = requests.get(raw_url, headers=headers, timeout=60)
    if resp.status_code == 404:
        raise FileNotFoundError(f"Not found: {path}")
    resp.raise_for_status()
    content = resp.text.strip()
    if not content:
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(content))


def file_exists_on_github(repo: str, path: str, token: str) -> bool:
    url  = f"{GITHUB_API}/repos/{repo}/contents/{path}"
    resp = requests.get(url, headers=_gh_headers(token), timeout=10)
    return resp.status_code == 200


print("GitHub helpers loaded ✓")

GitHub helpers loaded ✓


---
## 4. FULL HISTORICAL FETCH

> Fetches OHLCV for every ticker in the universe, applies the $20 price filter,
> and saves one CSV per ticker to `stocks/` in your GitHub repo.
> Skips tickers already stored — safe to re-run if interrupted.
> **Expect 30–90 minutes** for ~600 tickers.

In [5]:
!pip install yfinance
import yfinance as yf

if GITHUB_TOKEN is None:
    print("  GITHUB_TOKEN not set.")
else:
    tickers      = universe["ticker"].tolist()
    meta_lookup  = universe.set_index("ticker")[["name", "sector"]].to_dict("index")

    index_rows   = []   # master index entries
    skipped      = []   # below $20
    failed       = []   # API errors

    print(f"Fetching {len(tickers)} tickers ({START_DATE} → {END_DATE}) …\n")

    for i, ticker in enumerate(tickers, 1):
        path = f"{STOCKS_PREFIX}/prices_{ticker}.csv"

        # ── Skip if already on GitHub ──────────────────────────────────────
        if file_exists_on_github(GITHUB_REPO, path, GITHUB_TOKEN):
            print(f"  [{i}/{len(tickers)}] {ticker:6s} — already stored, skipping")
            # Still add to index so it appears in the master list
            try:
                df_existing = load_csv_from_github(GITHUB_REPO, path, GITHUB_TOKEN)
                if not df_existing.empty:
                    index_rows.append({
                        "ticker"       : ticker,
                        "name"         : meta_lookup.get(ticker, {}).get("name", ""),
                        "sector"       : meta_lookup.get(ticker, {}).get("sector", ""),
                        "max_price"    : round(df_existing["Close"].max(), 2),
                        "row_count"    : len(df_existing),
                        "start_date"   : df_existing["Date"].min(),
                        "last_updated" : datetime.now(timezone.utc).strftime("%Y-%m-%d"),
                    })
            except Exception:
                pass
            continue

        # ── Fetch from Yahoo Finance ───────────────────────────────────────
        try:
            df = yf.download(
                ticker,
                start    = START_DATE,
                end      = END_DATE,
                progress = False,
                auto_adjust = True,
            )

            if df.empty:
                print(f"  [{i}/{len(tickers)}] {ticker:6s} — no data")
                failed.append(ticker)
                continue

            # Flatten MultiIndex columns if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df = df.reset_index()
            df["ticker"] = ticker

            # ── Apply $20 price filter ─────────────────────────────────────
            max_price = df["Close"].max()
            if max_price < MIN_PRICE:
                print(f"  [{i}/{len(tickers)}] {ticker:6s} — max ${max_price:.2f} < ${MIN_PRICE} — skipped")
                skipped.append(ticker)
                continue

            # ── Push to GitHub ─────────────────────────────────────────────
            push_csv_to_github(
                df             = df,
                repo           = GITHUB_REPO,
                path           = path,
                token          = GITHUB_TOKEN,
                commit_message = f"Add {ticker} prices ({len(df)} rows, max ${max_price:.2f})",
            )

            index_rows.append({
                "ticker"       : ticker,
                "name"         : meta_lookup.get(ticker, {}).get("name", ""),
                "sector"       : meta_lookup.get(ticker, {}).get("sector", ""),
                "max_price"    : round(max_price, 2),
                "row_count"    : len(df),
                "start_date"   : str(df["Date"].min())[:10],
                "last_updated" : datetime.now(timezone.utc).strftime("%Y-%m-%d"),
            })
            print(f"  [{i}/{len(tickers)}] {ticker:6s} — {len(df)} rows, max ${max_price:.2f} ✓")

            time.sleep(0.3)   # be polite to Yahoo

        except Exception as e:
            print(f"  [{i}/{len(tickers)}] {str(ticker):6s} — ERROR: {e}")
            failed.append(ticker)
            continue

    # ── Push master index ──────────────────────────────────────────────────
    if index_rows:
        df_index = pd.DataFrame(index_rows).sort_values("ticker").reset_index(drop=True)
        push_csv_to_github(
            df             = df_index,
            repo           = GITHUB_REPO,
            path           = f"{STOCKS_PREFIX}/index.csv",
            token          = GITHUB_TOKEN,
            commit_message = f"Update stock index — {len(df_index)} tickers",
        )

    print(f"\n Done!")
    print(f"   Stored  : {len(index_rows)} tickers")
    print(f"   Skipped : {len(skipped)} tickers below ${MIN_PRICE}")
    print(f"   Failed  : {len(failed)} tickers — {failed}")

Fetching 5533 tickers (2010-01-01 → 2026-06-27) …

  [1/5533] AAAP   — 4 rows, max $25.16 ✓
  [2/5533] AACB   — max $10.55 < $20.0 — skipped
  [3/5533] AACBR  — max $0.21 < $20.0 — skipped
  [4/5533] AACBU  — max $12.79 < $20.0 — skipped
  [5/5533] AACG   — max $14.30 < $20.0 — skipped
  [6/5533] AACI   — max $9.98 < $20.0 — skipped
  [7/5533] AACIU  — max $10.30 < $20.0 — skipped
  [8/5533] AACIW  — max $0.65 < $20.0 — skipped
  [9/5533] AACO   — max $9.94 < $20.0 — skipped
  [10/5533] AACOU  — max $10.20 < $20.0 — skipped
  [11/5533] AACOW  — max $0.42 < $20.0 — skipped
  [12/5533] AACP   — max $10.00 < $20.0 — skipped
  [13/5533] AACPR  — max $0.21 < $20.0 — skipped
  [14/5533] AACPU  — max $10.35 < $20.0 — skipped
  [15/5533] AACPW  — max $0.13 < $20.0 — skipped
  [16/5533] AADR   — 4008 rows, max $96.52 ✓
  [17/5533] AAEQ   — 134 rows, max $54.13 ✓
  [18/5533] AAL    — 4145 rows, max $56.99 ✓
  [19/5533] AALG   — 242 rows, max $20.04 ✓
  [20/5533] AAME   — max $6.24 < $20.0 — skip

Failed to get ticker 'ALIS' reason: Failed to perform, curl: (28) Connection timed out after 39541 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.


  [231/5533] ALIS   — max $10.15 < $20.0 — skipped
  [232/5533] ALISR  — max $0.80 < $20.0 — skipped
  [233/5533] ALISU  — max $11.36 < $20.0 — skipped
  [234/5533] ALKS   — 4145 rows, max $80.14 ✓
  [235/5533] ALKT   — 1307 rows, max $47.69 ✓
  [236/5533] ALLO   — 1936 rows, max $54.04 ✓
  [237/5533] ALLR   — 1132 rows, max $10605000.00 ✓
  [238/5533] ALLT   — 4145 rows, max $28.03 ✓
  [239/5533] ALLW   — 329 rows, max $30.26 ✓
  [240/5533] ALM    — 3277 rows, max $23.42 ✓
  [241/5533] ALMR   — 49 rows, max $26.87 ✓
  [242/5533] ALMS   — 500 rows, max $29.98 ✓
  [243/5533] ALMU   — 835 rows, max $31.49 ✓
  [244/5533] ALNT   — 4145 rows, max $102.26 ✓
  [245/5533] ALNY   — already stored, skipping
  [246/5533] ALOT   — 4145 rows, max $28.57 ✓
  [247/5533] ALOV   — max $9.95 < $20.0 — skipped
  [248/5533] ALOVU  — max $10.24 < $20.0 — skipped
  [249/5533] ALOVW  — max $0.45 < $20.0 — skipped
  [250/5533] ALOY   — 2490 rows, max $63.00 ✓
  [251/5533] ALP    — 4145 rows, max $1236.00 ✓
  

$FBYDP: possibly delisted; no price data found  (1d 2010-01-01 -> 2026-06-27)

1 Failed download:
['FBYDP']: possibly delisted; no price data found  (1d 2010-01-01 -> 2026-06-27)


  [1726/5533] FBYDP  — no data
  [1727/5533] FBYDW  — max $0.98 < $20.0 — skipped
  [1728/5533] FBYY   — max $18.87 < $20.0 — skipped
  [1729/5533] FCA    — 3817 rows, max $34.11 ✓
  [1730/5533] FCAL   — 2262 rows, max $49.58 ✓
  [1731/5533] FCAP   — 4145 rows, max $68.91 ✓
  [1732/5533] FCBC   — 4145 rows, max $44.61 ✓
  [1733/5533] FCCO   — 4145 rows, max $32.74 ✓
  [1734/5533] FCEF   — 2448 rows, max $23.87 ✓
  [1735/5533] FCEL   — 4145 rows, max $17366.40 ✓
  [1736/5533] FCFS   — 4145 rows, max $232.57 ✓
  [1737/5533] FCHL   — 204 rows, max $3240.00 ✓
  [1738/5533] FCLO   — 93 rows, max $50.35 ✓
  [1739/5533] FCNCA  — 4145 rows, max $2336.57 ✓
  [1740/5533] FCNCN  — 102 rows, max $25.18 ✓
  [1741/5533] FCNCO  — 1120 rows, max $22.54 ✓
  [1742/5533] FCNCP  — 1585 rows, max $21.90 ✓
  [1743/5533] FCTE   — 498 rows, max $28.79 ✓
  [1744/5533] FCUV   — 2918 rows, max $1285.33 ✓
  [1745/5533] FCVT   — 2675 rows, max $54.63 ✓
  [1746/5533] FCXG   — max $18.07 < $20.0 — skipped
  [1747/55

...

  [3131/5533] MIDDV  — no data
  [3132/5533] MILN   — 2549 rows, max $50.61 ✓
  [3133/5533] MIMI   — 366 rows, max $107.50 ✓
  [3134/5533] MIND   — 4145 rows, max $260.90 ✓
  [3135/5533] MIRA   — max $7.42 < $20.0 — skipped
  [3136/5533] MIRM   — 1745 rows, max $117.28 ✓
  [3137/5533] MIST   — 1793 rows, max $27.15 ✓
  [3138/5533] MITK   — 4145 rows, max $23.03 ✓
  [3139/5533] MKAM   — 805 rows, max $32.19 ✓
  [3140/5533] MKDW   — 815 rows, max $345.90 ✓
  [3141/5533] MKDWW  — max $0.01 < $20.0 — skipped
  [3142/5533] MKLY   — max $10.19 < $20.0 — skipped
  [3143/5533] MKLYR  — max $0.23 < $20.0 — skipped
  [3144/5533] MKLYU  — max $10.33 < $20.0 — skipped
  [3145/5533] MKSI   — 4145 rows, max $420.56 ✓
  [3146/5533] MKTW   — 1453 rows, max $221.28 ✓
  [3147/5533] MKTX   — 4145 rows, max $553.43 ✓
  [3148/5533] MKZR   — 494 rows, max $46.92 ✓
  [3149/5533] MLAA   — max $10.14 < $20.0 — skipped
  [3150/5533] MLAAU  — max $10.40 < $20.0 — skipped
  [3151/5533] MLAAW  — max $0.27 < $20.0 

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ZAZZT"}}}
$ZAZZT: possibly delisted; no timezone found

1 Failed download:
['ZAZZT']: possibly delisted; no timezone found


  [5487/5533] ZAZZT  — no data
  [5488/5533] ZBAO   — max $4.61 < $20.0 — skipped
  [5489/5533] ZBIO   — 447 rows, max $43.17 ✓
  [5490/5533] ZBRA   — already stored, skipping
  [5491/5533] ZBZZT  — 1 rows, max $199999.98 ✓
  [5492/5533] ZCMD   — 1593 rows, max $10242.40 ✓


$ZCZZT: possibly delisted; no timezone found

1 Failed download:
['ZCZZT']: possibly delisted; no timezone found


  [5493/5533] ZCZZT  — no data
  [5494/5533] ZD     — 4145 rows, max $132.98 ✓
  [5495/5533] ZDAI   — 484 rows, max $1717.76 ✓
  [5496/5533] ZENA   — max $10.30 < $20.0 — skipped
  [5497/5533] ZEO    — max $11.52 < $20.0 — skipped
  [5498/5533] ZEOWW  — max $0.04 < $20.0 — skipped
  [5499/5533] ZG     — 3756 rows, max $203.79 ✓
  [5500/5533] ZHOG   — 704 rows, max $51.47 ✓
  [5501/5533] ZION   — 4145 rows, max $69.33 ✓
  [5502/5533] ZIONP  — 4145 rows, max $22.66 ✓
  [5503/5533] ZJK    — max $17.26 < $20.0 — skipped
  [5504/5533] ZJYL   — 815 rows, max $276.11 ✓
  [5505/5533] ZJZZT  — max $10.02 < $20.0 — skipped
  [5506/5533] ZKIN   — max $12.35 < $20.0 — skipped
  [5507/5533] ZKP    — max $10.01 < $20.0 — skipped
  [5508/5533] ZKPU   — max $10.15 < $20.0 — skipped
  [5509/5533] ZKPW   — max $0.33 < $20.0 — skipped
  [5510/5533] ZLAB   — 2203 rows, max $191.71 ✓
  [5511/5533] ZM     — 1807 rows, max $568.34 ✓
  [5512/5533] ZMUN   — 186 rows, max $50.21 ✓
  [5513/5533] ZNB    — 4145 ro

---
## 5. Load a single ticker

In [6]:
TICKER = "NVDA"   # ← change this

df_ticker = load_csv_from_github(
    GITHUB_REPO,
    f"{STOCKS_PREFIX}/prices_{TICKER}.csv",
    GITHUB_TOKEN,
)
df_ticker["Date"] = pd.to_datetime(df_ticker["Date"])
print(f"{TICKER}: {len(df_ticker)} rows, {df_ticker['Date'].min().date()} → {df_ticker['Date'].max().date()}")
df_ticker.tail()

NVDA: 4145 rows, 2010-01-04 → 2026-06-26


,Date,Close,High,Low,Open,Volume,ticker
4140,2026-06-22,208.649994,213.990005,207.720001,211.440002,122041400,NVDA
4141,2026-06-23,200.039993,203.770004,200.000000,202.169998,153496200,NVDA
4142,2026-06-24,199.000000,201.669998,196.580002,200.119995,151810700,NVDA
4143,2026-06-25,195.740005,200.800003,192.130005,200.080002,149550000,NVDA
4144,2026-06-26,NaN,NaN,NaN,NaN,177757849,NVDA


---
## 6. Load the master index

In [7]:
df_index = load_csv_from_github(
    GITHUB_REPO,
    f"{STOCKS_PREFIX}/index.csv",
    GITHUB_TOKEN,
)
print(f"{len(df_index)} tickers in index")
print(f"Sectors: {df_index['sector'].value_counts().to_dict()}")
df_index.sort_values("max_price", ascending=False).head(20)

3851 tickers in index
Sectors: {'NASDAQ': 3851}


,ticker,name,sector,max_price,row_count,start_date,last_updated
3795,XTIA,"XTI Aerospace, Inc. Common Stock - Common Stock",NASDAQ,1.974375e+13,3493,2012-08-03,2026-06-28
2668,PPCB,"Propanc Biopharma, Inc. - Common Stock",NASDAQ,6.250000e+12,3388,2013-01-07,2026-06-27
2194,MRDN,Meridian Holdings Inc. - Common Stock,NASDAQ,1.636200e+12,4145,2010-01-04,2026-06-27
2389,NUWE,"Nuwellis, Inc. - Common Stock",NASDAQ,7.216171e+11,3610,2012-02-16,2026-06-27
1821,JAGX,"Jaguar Health, Inc. - Common Stock",NASDAQ,8.694591e+10,2798,2015-05-12,2026-06-27
3693,WHLR,"Wheeler Real Estate Investment Trust, Inc. - C...",NASDAQ,6.522079e+10,3420,2012-11-19,2026-06-28
3363,TNXP,Tonix Pharmaceuticals Holding Corp. - Common S...,NASDAQ,1.920000e+10,3552,2012-05-10,2026-06-27
1057,EJH,E-Home Household Service Holdings Limited - Or...,NASDAQ,6.750000e+09,1285,2021-05-14,2026-06-27
3133,SMX,SMX (Security Matters) Public Limited Company ...,NASDAQ,2.444036e+09,1140,2021-12-09,2026-06-27
673,CENN,Cenntro Inc. - Common Stock,NASDAQ,2.430000e+09,3457,2012-09-25,2026-06-27


---
## 7. Incremental update — refresh all stored tickers

In [4]:
import yfinance as yf

if GITHUB_TOKEN is None:
    print("  GITHUB_TOKEN not set.")
else:
    # Load the index to know which tickers we have
    df_index = load_csv_from_github(GITHUB_REPO, f"{STOCKS_PREFIX}/index.csv", GITHUB_TOKEN)
    tickers  = df_index["ticker"].tolist()
    today    = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    print(f"Updating {len(tickers)} tickers through {today} …\n")
    updated = 0

    for i, ticker in enumerate(tickers, 1):
        if not isinstance(ticker, str) or ticker.lower() == "nan":
            continue
        path = f"{STOCKS_PREFIX}/prices_{ticker}.csv"
        try:
            df_existing = load_csv_from_github(GITHUB_REPO, path, GITHUB_TOKEN)
            df_existing["Date"] = pd.to_datetime(df_existing["Date"])
            last_date = df_existing["Date"].max()
            fetch_from = (last_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

            if fetch_from >= today:
                print(f"  [{i}/{len(tickers)}] {ticker:6s} — already current")
                continue

            df_new = yf.download(ticker, start=fetch_from, end=today, progress=False, auto_adjust=True)
            if df_new.empty:
                print(f"  [{i}/{len(tickers)}] {ticker:6s} — no new data")
                continue

            if isinstance(df_new.columns, pd.MultiIndex):
                df_new.columns = df_new.columns.get_level_values(0)
            df_new = df_new.reset_index()
            df_new["ticker"] = ticker

            df_combined = (
                pd.concat([df_existing, df_new], ignore_index=True)
                .drop_duplicates(subset="Date")
                .sort_values("Date")
                .reset_index(drop=True)
            )

            push_csv_to_github(
                df             = df_combined,
                repo           = GITHUB_REPO,
                path           = path,
                token          = GITHUB_TOKEN,
                commit_message = f"Update {ticker} +{len(df_new)} rows through {today}",
            )
            print(f"  [{i}/{len(tickers)}] {ticker:6s} — +{len(df_new)} new rows ✓")
            updated += 1
            time.sleep(0.3)

        except Exception as e:
            print(f"  [{i}/{len(tickers)}] {str(ticker):6s} — ERROR: {e}")

    print(f"\n✓ Updated {updated} tickers")

Updating 3851 tickers through 2026-06-28 …

  [1/3851] AAAP   — +1 new rows ✓
  [2/3851] AADR   — +1 new rows ✓
  [3/3851] AAEQ   — +1 new rows ✓
  [4/3851] AAL    — +1 new rows ✓
  [5/3851] AALG   — +1 new rows ✓
  [6/3851] AAOI   — +1 new rows ✓
  [7/3851] AAON   — +1 new rows ✓
  [8/3851] AAPB   — +1 new rows ✓
  [9/3851] AAPD   — +1 new rows ✓
  [10/3851] AAPG   — +1 new rows ✓
  [11/3851] AAPL   — +1 new rows ✓
  [12/3851] AAPU   — +1 new rows ✓
  [13/3851] AAUS   — +1 new rows ✓
  [14/3851] AAVM   — +1 new rows ✓
  [15/3851] AAXJ   — +1 new rows ✓
  [16/3851] ABAT   — +1 new rows ✓
  [17/3851] ABCL   — +1 new rows ✓
  [18/3851] ABCS   — +1 new rows ✓
  [19/3851] ABEO   — +1 new rows ✓
  [20/3851] ABI    — +1 new rows ✓
  [21/3851] ABIG   — +1 new rows ✓
  [22/3851] ABNB   — +1 new rows ✓
  [23/3851] ABOS   — +1 new rows ✓
  [24/3851] ABSI   — +1 new rows ✓
  [25/3851] ABTC   — +1 new rows ✓
  [26/3851] ABTS   — +1 new rows ✓
  [27/3851] ABUS   — +1 new rows ✓
  [28/3851] ABVC   —

$AFJKU: possibly delisted; no price data found  (1d 2026-06-27 -> 2026-06-28)

1 Failed download:
['AFJKU']: possibly delisted; no price data found  (1d 2026-06-27 -> 2026-06-28)


  [88/3851] AFJKU  — no new data
  [89/3851] AFOS   — +1 new rows ✓
  [90/3851] AFRM   — +1 new rows ✓
  [91/3851] AFSC   — +1 new rows ✓
  [92/3851] AFYA   — +1 new rows ✓
  [93/3851] AGCC   — +1 new rows ✓
  [94/3851] AGEM   — +1 new rows ✓
  [95/3851] AGEN   — +1 new rows ✓
  [96/3851] AGGA   — +1 new rows ✓
  [97/3851] AGIO   — +1 new rows ✓
  [98/3851] AGIX   — +1 new rows ✓
  [99/3851] AGMH   — +1 new rows ✓
  [100/3851] AGMI   — +1 new rows ✓
  [101/3851] AGNCL  — +1 new rows ✓
  [102/3851] AGNCM  — +1 new rows ✓
  [103/3851] AGNCN  — +1 new rows ✓
  [104/3851] AGNCO  — +1 new rows ✓
  [105/3851] AGNCP  — +1 new rows ✓
  [106/3851] AGNCZ  — +1 new rows ✓
  [107/3851] AGNG   — +1 new rows ✓
  [108/3851] AGNT   — +1 new rows ✓
  [109/3851] AGPU   — +1 new rows ✓
  [110/3851] AGYS   — +1 new rows ✓
  [111/3851] AGZD   — +1 new rows ✓
  [112/3851] AHCO   — +1 new rows ✓
  [113/3851] AHD    — +1 new rows ✓
  [114/3851] AHG    — +1 new rows ✓
  [115/3851] AHMA   — +1 new rows ✓
  [116

$ASPC: possibly delisted; no price data found  (1d 2026-06-27 -> 2026-06-28)

1 Failed download:
['ASPC']: possibly delisted; no price data found  (1d 2026-06-27 -> 2026-06-28)


  [284/3851] ASPC   — no new data


...

  [1951/3851] LBTYB  — no new data
  [1952/3851] LBTYK  — +1 new rows ✓
  [1953/3851] LCDL   — +1 new rows ✓
  [1954/3851] LCDS   — +1 new rows ✓
  [1955/3851] LCFY   — +1 new rows ✓
  [1956/3851] LCID   — +1 new rows ✓
  [1957/3851] LDEM   — +1 new rows ✓
  [1958/3851] LDRX   — +1 new rows ✓
  [1959/3851] LE     — +1 new rows ✓
  [1960/3851] LECO   — +1 new rows ✓
  [1961/3851] LEDS   — +1 new rows ✓
  [1962/3851] LEE    — +1 new rows ✓
  [1963/3851] LEGH   — +1 new rows ✓
  [1964/3851] LEGN   — +1 new rows ✓
  [1965/3851] LEGR   — +1 new rows ✓
  [1966/3851] LEND   — +1 new rows ✓
  [1967/3851] LENZ   — +1 new rows ✓
  [1968/3851] LESL   — +1 new rows ✓
  [1969/3851] LEXI   — +1 new rows ✓
  [1970/3851] LFMD   — +1 new rows ✓
  [1971/3851] LFMDP  — +1 new rows ✓
  [1972/3851] LFSC   — +1 new rows ✓
  [1973/3851] LFST   — +1 new rows ✓
  [1974/3851] LFTO   — +1 new rows ✓
  [1975/3851] LFUS   — +1 new rows ✓
  [1976/3851] LFVN   — +1 new rows ✓
  [1977/3851] LFWD   — +1 new rows ✓
  [

$MCTA: possibly delisted; no price data found  (1d 2026-06-27 -> 2026-06-28)

1 Failed download:
['MCTA']: possibly delisted; no price data found  (1d 2026-06-27 -> 2026-06-28)


  [2103/3851] MCTA   — no new data
  [2104/3851] MDB    — +1 new rows ✓
  [2105/3851] MDGL   — +1 new rows ✓
  [2106/3851] MDLN   — +1 new rows ✓
  [2107/3851] MDLZ   — +1 new rows ✓
  [2108/3851] MDRR   — +1 new rows ✓
  [2109/3851] MDWD   — +1 new rows ✓
  [2110/3851] MEDP   — +1 new rows ✓
  [2111/3851] MEDX   — +1 new rows ✓
  [2112/3851] MEGL   — +1 new rows ✓
  [2113/3851] MELI   — +1 new rows ✓
  [2114/3851] MEMA   — +1 new rows ✓
  [2115/3851] MEMS   — +1 new rows ✓
  [2116/3851] MENS   — +1 new rows ✓
  [2117/3851] MEOH   — +1 new rows ✓
  [2118/3851] MESO   — +1 new rows ✓
  [2119/3851] META   — +1 new rows ✓
  [2120/3851] METC   — +1 new rows ✓
  [2121/3851] METCB  — +1 new rows ✓
  [2122/3851] METCI  — +1 new rows ✓
  [2123/3851] METCZ  — +1 new rows ✓
  [2124/3851] METD   — +1 new rows ✓
  [2125/3851] METL   — +1 new rows ✓
  [2126/3851] METU   — +1 new rows ✓
  [2127/3851] MFI    — +1 new rows ✓
  [2128/3851] MFICL  — +1 new rows ✓
  [2129/3851] MFIG   — +1 new rows ✓
  [

$RDIB: possibly delisted; no price data found  (1d 2026-06-27 -> 2026-06-28)

1 Failed download:
['RDIB']: possibly delisted; no price data found  (1d 2026-06-27 -> 2026-06-28)


  [2861/3851] RDIB   — no new data
  [2862/3851] RDNT   — +1 new rows ✓
  [2863/3851] RDNW   — +1 new rows ✓
  [2864/3851] RDTL   — +1 new rows ✓
  [2865/3851] RDTY   — +1 new rows ✓
  [2866/3851] RDVT   — +1 new rows ✓
  [2867/3851] RDVY   — +1 new rows ✓
  [2868/3851] RDWR   — +1 new rows ✓
  [2869/3851] REAI   — +1 new rows ✓
  [2870/3851] REAL   — +1 new rows ✓
  [2871/3851] REBN   — +1 new rows ✓
  [2872/3851] REE    — +1 new rows ✓
  [2873/3851] REG    — +1 new rows ✓
  [2874/3851] REGCO  — +1 new rows ✓
  [2875/3851] REGCP  — +1 new rows ✓
  [2876/3851] REGN   — +1 new rows ✓
  [2877/3851] REIT   — +1 new rows ✓
  [2878/3851] REKR   — +1 new rows ✓
  [2879/3851] RELL   — +1 new rows ✓
  [2880/3851] RELY   — +1 new rows ✓
  [2881/3851] REMG   — +1 new rows ✓
  [2882/3851] RENT   — +1 new rows ✓
  [2883/3851] RENX   — +1 new rows ✓
  [2884/3851] REPL   — +1 new rows ✓
  [2885/3851] RETO   — +1 new rows ✓
  [2886/3851] REVB   — +1 new rows ✓
  [2887/3851] REXC   — +1 new rows ✓
  [

---
## 8. Quick exploration — your portfolio tickers

In [7]:
import warnings

missing = ["KGC", "NVO", "TSM", "WPM"]

for ticker in missing:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index()
    df["ticker"] = ticker
    max_price = df["Close"].max()
    push_csv_to_github(
        df             = df,
        repo           = GITHUB_REPO,
        path           = f"{STOCKS_PREFIX}/prices_{ticker}.csv",
        token          = GITHUB_TOKEN,
        commit_message = f"Add {ticker} prices ({len(df)} rows, max ${max_price:.2f})",
    )
    print(f"✓ {ticker} — {len(df)} rows, max ${max_price:.2f}")

✓ KGC — 4145 rows, max $37.97
✓ NVO — 4145 rows, max $137.40
✓ TSM — 4145 rows, max $467.67
✓ WPM — 4145 rows, max $165.21


In [6]:
print("finished")

finished
